In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import yfinance

In [ ]:
# Constants
RISK_FREE_RATE = 0.042

INDICES = ["SPY", "IWM", "DIA"]
TICKERS = ["MU", "EQIX", "ETN", "AMD", "EL", "WDC", "ANET"]

PERIOD = "10y"

In [ ]:
# Download ticker data
yfinance_data = yfinance.download(tickers=TICKERS+INDICES, period=PERIOD)['Close']

In [ ]:
def calculate_returns(prices: pd.DataFrame) -> pd.DataFrame:
    daily_rets = prices.pct_change().dropna()
    return daily_rets

def calculate_vol(returns: pd.DataFrame) -> float:
    annualized_vol = np.sqrt(252) * returns.std()
    return annualized_vol

def calculate_beta(x: pd.Series, y: pd.Series):
    aligned = pd.concat([y, x], axis=1).dropna()
    if aligned.shape[0] < 2:
        return np.nan
    cov = np.cov(aligned.iloc[:,0], aligned.iloc[:,1], ddof=1)[0,1]
    var = np.var(aligned.iloc[:,1], ddof=1)
    return cov / var if var != 0 else np.nan


def weekly_drawdowns(prices: pd.DataFrame, weeks: int = 52):
    week = prices.resample('W').agg(['max', 'min']).dropna()
    week['drawdown'] = (week['min'] - week['max']) / week['max']
    last = week.tail(weeks)
    if last.empty:
        return np.nan, np.nan
    avg = last['drawdown'].mean()
    max = last['drawdown'].min()
    return avg, max

def total_return(prices: pd.DataFrame, years: int = 10):
    end_date = prices.dropna().index.max()
    start_date = end_date - pd.DateOffset(years=years)
    window = prices.loc[start_date:end_date].dropna()
    if window.empty or window.shape[0] < 2:
        return np.nan, np.nan
    tr = window.iloc[-1] / window.iloc[0] - 1.0
    ar = (1.0 + tr) ** (1/years) - 1.0
    return tr, ar
